# 1. Get the data

This notebook downloads the raw data and saves it in `data/raw/`:

- 14 currency pairs from Yahoo Finance (2005 onward)
- Daily exchange rates against the US dollar from FRED (1999 onward), for the training history
- The S&P 500 from Yahoo Finance (1999 onward)
- 3-month and 10-year interest rates for seven currencies, two overnight rates, and the VIX, from FRED

The download date is saved at the end, because Yahoo sometimes revises past data.

In [ ]:
# JupyterLab on Nuvolos starts the kernel in /files, while VS Code starts it in the notebook's
# folder. The notebooks need to run from code/ to find helpers.py and ../data, so this cell
# looks for that folder and moves into it.
import os
from pathlib import Path

if not Path("helpers.py").exists():
    found = [p.parent for p in Path.cwd().glob("**/code/helpers.py")
             if (p.parent / "05_figures.ipynb").exists()]
    if len(found) != 1:
        raise FileNotFoundError(f"Cannot find a single project code folder under {Path.cwd()}: {found}")
    os.chdir(found[0])
print("Working directory:", Path.cwd())

In [1]:
import os
from datetime import date

import pandas as pd
import yfinance as yf

import helpers

os.makedirs(helpers.RAW, exist_ok=True)

## 1. Currency pairs (Yahoo)

In [2]:
for p in helpers.ALL_PAIRS:
    df = yf.download(p + "=X", start=helpers.START, progress=False)
    if len(df) == 0:
        print(p, "NO DATA")
    else:
        df.to_csv(f"{helpers.RAW}/{p}.csv")
        print(p, len(df), "rows,", df.index.min().date(), "to", df.index.max().date())

CADJPY 5690 rows, 2005-01-03 to 2026-11-13
AUDUSD 5333 rows, 2006-05-16 to 2026-11-16
USDJPY 5672 rows, 2005-01-03 to 2026-11-13
EURUSD 5673 rows, 2005-01-03 to 2026-11-16
EURCHF 5674 rows, 2005-01-03 to 2026-11-16
GBPJPY 5689 rows, 2005-01-03 to 2026-11-16
USDCAD 5687 rows, 2005-01-03 to 2026-11-13
GBPUSD 5685 rows, 2005-01-03 to 2026-11-16
EURGBP 5693 rows, 2005-01-03 to 2026-11-16
GBPCHF 5687 rows, 2005-01-03 to 2026-11-13
CHFJPY 5690 rows, 2005-01-03 to 2026-11-13
AUDJPY 5691 rows, 2005-01-03 to 2026-11-13
EURJPY 5676 rows, 2005-01-03 to 2026-11-16
USDCHF 5685 rows, 2005-01-03 to 2026-11-13


## 2. Daily exchange rates against the dollar (FRED, from 1999)

In [3]:
def download_fred(code):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    return pd.read_csv(url, index_col=0, parse_dates=True)

for pair, code in helpers.FRED_FX.items():
    df = download_fred(code)
    df = df[df.index >= helpers.HISTORY_START]
    df.to_csv(f"{helpers.RAW}/FREDFX_{pair}.csv")
    values = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    print(pair, code, len(values), "days,", values.index.min().date(), "to", values.index.max().date())

EURUSD DEXUSEU 6984 days, 1999-01-04 to 2026-11-06
USDJPY DEXJPUS 6984 days, 1999-01-04 to 2026-11-06
USDCHF DEXSZUS 6984 days, 1999-01-04 to 2026-11-06
GBPUSD DEXUSUK 6984 days, 1999-01-04 to 2026-11-06
AUDUSD DEXUSAL 6984 days, 1999-01-04 to 2026-11-06
USDCAD DEXCAUS 6984 days, 1999-01-04 to 2026-11-06


## 3. S&P 500 (from 1999, the investor's portfolio)

In [4]:
sp = yf.download("^GSPC", start=helpers.HISTORY_START, progress=False)
sp.to_csv(f"{helpers.RAW}/SP500.csv")
print("SP500", len(sp), "rows,", sp.index.min().date(), "to", sp.index.max().date())

SP500 7011 rows, 1999-01-04 to 2026-11-16


## 4. Interest rates and the VIX (FRED)

In [5]:
for name, code in {**helpers.FRED_SERIES, **helpers.FRED_BACKFILL}.items():
    df = download_fred(code)
    df.to_csv(f"{helpers.RAW}/FRED_{name}.csv")
    values = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    print(name, code, values.index.min().date(), "to", values.index.max().date())

USD_3M IR3TIB01USM156N 1964-06-01 to 2026-09-01
USD_10Y IRLTLT01USM156N 1953-04-01 to 2026-09-01
EUR_3M IR3TIB01EZM156N 1994-01-01 to 2026-01-01
EUR_10Y IRLTLT01EZM156N 1970-01-01 to 2026-01-01
GBP_3M IR3TIB01GBM156N 1957-01-01 to 2026-01-01
GBP_10Y IRLTLT01GBM156N 1960-01-01 to 2026-09-01
JPY_3M IR3TIB01JPM156N 2002-04-01 to 2026-08-01
JPY_10Y IRLTLT01JPM156N 1989-01-01 to 2026-09-01
CHF_3M IR3TIB01CHM156N 1999-07-01 to 2026-09-01
CHF_10Y IRLTLT01CHM156N 1955-01-01 to 2026-09-01
AUD_3M IR3TIB01AUM156N 1968-01-01 to 2026-08-01
AUD_10Y IRLTLT01AUM156N 1969-07-01 to 2026-09-01
CAD_3M IR3TIB01CAM156N 1956-01-01 to 2026-09-01
CAD_10Y IRLTLT01CAM156N 1955-01-01 to 2026-09-01
VIX VIXCLS 1990-01-02 to 2026-11-13
CHF_CALL IRSTCI01CHM156N 1972-01-01 to 2024-03-01
JPY_CALL IRSTCI01JPM156N 1985-07-01 to 2026-09-01


## 5. Record the download date

In [6]:
with open(f"{helpers.RAW}/download_date.txt", "w") as f:
    f.write(str(date.today()))

print("Downloaded on", date.today())

Downloaded on 2026-11-16


## 6. Checks

In [7]:
files = os.listdir(helpers.RAW)
expected = (len(helpers.ALL_PAIRS) + len(helpers.FRED_FX) + 1
            + len(helpers.FRED_SERIES) + len(helpers.FRED_BACKFILL) + 1)
print(len(files), "files in data/raw, expected", expected)
assert len(files) == expected, "Some files are missing"

# The FRED history must start in January 1999
for pair in helpers.FRED_FX:
    first = pd.read_csv(f"{helpers.RAW}/FREDFX_{pair}.csv", index_col=0, parse_dates=True).index.min()
    assert first < pd.Timestamp("1999-01-10"), f"FRED {pair} starts too late"

print("All checks passed")

39 files in data/raw, expected 39
All checks passed
